# 使用新模板创建和评估因子

本 Notebook 演示：创建 `factor_analyse/factor_mining/` 下的新模板因子、修改公式、执行评估、读取保存结果和生成 HTML 报告。

请先运行创建单元格，再编辑生成的 `.py` 文件，最后运行评估单元格。

## 1. 初始化路径和环境

In [1]:
from pathlib import Path
import os
import sys

def find_repo_root(start: Path) -> Path:
    candidates = [start.resolve(), *start.resolve().parents]
    injected = os.environ.get("FACTOR_COMMON_REPO_ROOT")
    if injected:
        candidates.insert(0, Path(injected).expanduser().resolve())
    for candidate in candidates:
        if (candidate / "factor_common").is_dir() and (candidate / "data").is_dir():
            return candidate
    raise RuntimeError("无法定位 cryptoFactorAnalyze 仓库根目录")

ROOT = find_repo_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("repo root:", ROOT)

repo root: /Users/dmiwu/work/PythonProject/cryptoFactorAnalyze


In [2]:
from factor_common import FactorManager

FACTOR_NAME = "Volume_Stability_Factor"
H5_PATH = ROOT / "data" / "crypto_quant.h5"
FACTOR_DIR = ROOT / "factor_analyse" / "factor_mining"

manager = FactorManager(
    h5_path=H5_PATH,
    base_dir=ROOT / "data" / "factor_results",
    reports_dir=ROOT / "reports",
)

print("H5:", manager.h5_path)
print("factor directory:", FACTOR_DIR)

H5: /Users/dmiwu/work/PythonProject/cryptoFactorAnalyze/data/crypto_quant.h5
factor directory: /Users/dmiwu/work/PythonProject/cryptoFactorAnalyze/factor_analyse/factor_mining


## 2. 创建模板因子

该单元格不会覆盖已有文件。第一次运行会创建 `my_factor.py`；如果文件已经存在，则直接使用它。

In [4]:
factor_path = FACTOR_DIR / f"{FACTOR_NAME}.py"
if not factor_path.is_file():
    raise FileNotFoundError(f"未找到已改造的因子文件: {factor_path}")
print("当前使用的因子文件：", factor_path)
# print(factor_path.read_text(encoding="utf-8"))

当前使用的因子文件： /Users/dmiwu/work/PythonProject/cryptoFactorAnalyze/factor_analyse/factor_mining/Volume_Stability_Factor.py


## 3. 编辑因子公式

打开并编辑下面的文件：

```text
factor_analyse/factor_mining/my_factor.py
```

修改 `SETTING["data_needed"]`、`SETTING["params"]` 和 `calc_factor()`。因子值只能使用当前及历史数据；不要使用负向 `shift`、居中 rolling 或反向填充。编辑完成后回到本 Notebook 继续运行。

## 4. 执行因子评估

In [ ]:
result = manager.evaluate(
    factor_path,
    params={
        "start": "2024-01-01",
        "end": "2026-09-01",
        "rebalance_days": 1, 
        "n_groups": 10,
        "fee_rate": 0.0005,
        "slippage": 0.001,
        "include_funding": True,
    },
    plot=False,
)

print("status:", result["status"])
print("run_id:", result["run_id"])
print("evaluation_id:", result["evaluation_id"])
print("factor value shape:", result["factor_value"].shape)
print("report:", result["paths"]["report_path"])

status: incomplete
run_id: c7f41c18e3d1a42d
evaluation_id: None
factor value shape: (978, 157)
report: None


## 5. 查看因子值和 IC

In [6]:
factor_value = result["factor_value"]
display(factor_value.tail())

full_ic = result["factor_performance"]["samples"]["full"]["ic"]
print({
    key: full_ic[key]
    for key in ("ic_mean", "rank_ic_mean", "icir", "t_stat", "n_dates")
})

,1INCHUSDT,2ZUSDT,AAVEUSDT,ADAUSDT,AGIXUSDT,ALGOUSDT,APEUSDT,APTUSDT,ARBUSDT,ARUSDT,...,XLMUSDT,XMRUSDT,XPLUSDT,XRPUSDT,XTZUSDT,ZECUSDT,ZKJUSDT,ZKUSDT,ZROUSDT,币安人生USDT
date,,,,,,,,,,,,,,,,,,,,,
2026-08-31,NaN,NaN,0.660377,0.735849,NaN,0.396226,NaN,0.622642,-0.547170,NaN,...,0.773585,-0.849057,NaN,0.169811,NaN,0.584906,NaN,NaN,NaN,-0.396226
2026-09-01,NaN,NaN,0.622642,0.698113,NaN,0.358491,NaN,0.773585,-0.886792,NaN,...,0.735849,-0.811321,NaN,0.207547,NaN,0.584906,NaN,NaN,NaN,-0.396226
2026-09-02,NaN,NaN,0.673469,0.795918,NaN,0.387755,NaN,NaN,-0.918367,NaN,...,0.714286,-0.836735,NaN,0.224490,NaN,0.632653,NaN,NaN,NaN,NaN
2026-09-03,NaN,NaN,0.714286,0.755102,NaN,0.224490,NaN,NaN,-1.000000,NaN,...,0.632653,-0.714286,NaN,0.306122,NaN,0.673469,NaN,NaN,NaN,NaN
2026-09-04,NaN,NaN,0.714286,0.632653,NaN,-0.061224,NaN,NaN,-1.000000,NaN,...,0.755102,0.387755,NaN,0.265306,NaN,0.591837,NaN,NaN,NaN,NaN


{'ic_mean': -0.019057208204598667, 'rank_ic_mean': 0.0021918585018983216, 'icir': -0.11847000067101322, 't_stat': -3.7011211373817683, 'n_dates': 976}


## 6. 读取已保存因子值并生成报告

In [7]:
reloaded_value = manager.get_value(FACTOR_NAME, run_id=result["run_id"])
print("reload equals computed value:", reloaded_value.equals(factor_value))

saved_result = result

report_info = manager.plot_result(
    saved_result,
    output_path=ROOT / "reports" / f"{FACTOR_NAME}_reloaded.html",
)
print("reloaded report:", report_info["output_path"])

reload equals computed value: True
reloaded report: /Users/dmiwu/work/PythonProject/cryptoFactorAnalyze/reports/Volume_Stability_Factor_reloaded.html


## 7. 查看未来函数检查和资金覆盖诊断

In [8]:
validation = result["diagnostics"]["validation"]
print("static scan:", validation["static_scan"]["status"])
print("cutoff replay:", validation["cutoff"]["status"])
print("cutoff diffs:", [
    item["max_abs_diff"]
    for item in validation["cutoff"].get("cutoffs", [])
])
print("funding coverage:", result["diagnostics"]["coverage"]["funding"]["status_counts"])

static scan: clean
cutoff replay: verified
cutoff diffs: [0.0, 0.0]
funding coverage: {'complete': 10, 'unknown': 3}
